# Install Libraries

In [1]:
!pip install -q transformers datasets evaluate accelerate sentencepiece sacrebleu rouge_score bert_score nltk pandas scikit-learn openpyxl

  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 3.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 100.8/100.8 kB 3.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.1/61.1 kB 2.8 MB/s eta 0:00:00


# 2: Import libraries

In [2]:
import os
import re
import gc
import torch
import random
import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split
from datasets import Dataset, DatasetDict

from transformers import (
    AutoTokenizer,
    AutoModelForSeq2SeqLM,
    DataCollatorForSeq2Seq,
    Seq2SeqTrainer,
    Seq2SeqTrainingArguments,
    EarlyStoppingCallback
)

import evaluate
import nltk
from nltk.translate.bleu_score import sentence_bleu, SmoothingFunction
from nltk.translate.meteor_score import meteor_score

nltk.download("wordnet")
nltk.download("omw-1.4")

SEED = 42

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

device = "cuda" if torch.cuda.is_available() else "cpu"

print("GPU available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

[nltk_data] Downloading package wordnet to /root/nltk_data...
[nltk_data] Downloading package omw-1.4 to /root/nltk_data...


GPU available: True
GPU: Tesla T4


# 3: Upload dataset

In [3]:
from google.colab import files

uploaded = files.upload()
csv_path = list(uploaded.keys())[0]

print("Uploaded file:", csv_path)

Saving Counterspeech_dataset_Old_cleaned.csv to Counterspeech_dataset_Old_cleaned.csv
Uploaded file: Counterspeech_dataset_Old_cleaned.csv


# Load Dataset

In [4]:
df = pd.read_csv(csv_path)

print("Dataset shape:", df.shape)
print("Columns:", df.columns.tolist())

df.head()

Dataset shape: (2506, 5)
Columns: ['id', 'offensive_text', 'counterspeech_text', 'counterspeech_type', 'text_type']


,id,offensive_text,counterspeech_text,counterspeech_type,text_type
0,1,"কি মনে করছ, মেয়েরা কিছু বোঝে ?",এই কথা ঠিক না। বাংলাদেশে অনেক মেয়ে দক্ষতার সাথ...,fact_based,hate_speech
1,2,"তুই তো বেশ্যা, রাস্তায় খেপ মাইরা বেরাছ",মানুষ কে সম্মান করতে শিখুন। তবেই নিজে সম্মানিত...,warning_of_consequences,sexual_harassment
2,3,হিন্দুরাই দেশের সম্পদ পাচার করতেছে,পুরো একটা গোষ্ঠীকে দোষ দেওয়া ঠিক না।,empathy,hate_speech
3,4,মাইয়া মানুষ আবার গাড়ি চালাইতে পারে নাকি!,এই ধারণা ভুল। বর্তমানে অনেক মেয়েরা সফলভাবে গাড়...,fact_based,gender_abuse
4,5,এই ছাগলের কথা শুনলে মাথাই গরম হয়ে যায়,ভদ্রভাবে কথা বললে আলোচনা আরও সুন্দর হয়।,moral_appeal,abusive_language


# 5: Keep only generation columns

In [5]:
required_columns = ["offensive_text", "counterspeech_text"]

for col in required_columns:
    if col not in df.columns:
        raise ValueError(f"Missing required column: {col}")

df = df[required_columns].copy()

df.head()

,offensive_text,counterspeech_text
0,"কি মনে করছ, মেয়েরা কিছু বোঝে ?",এই কথা ঠিক না। বাংলাদেশে অনেক মেয়ে দক্ষতার সাথ...
1,"তুই তো বেশ্যা, রাস্তায় খেপ মাইরা বেরাছ",মানুষ কে সম্মান করতে শিখুন। তবেই নিজে সম্মানিত...
2,হিন্দুরাই দেশের সম্পদ পাচার করতেছে,পুরো একটা গোষ্ঠীকে দোষ দেওয়া ঠিক না।
3,মাইয়া মানুষ আবার গাড়ি চালাইতে পারে নাকি!,এই ধারণা ভুল। বর্তমানে অনেক মেয়েরা সফলভাবে গাড়...
4,এই ছাগলের কথা শুনলে মাথাই গরম হয়ে যায়,ভদ্রভাবে কথা বললে আলোচনা আরও সুন্দর হয়।


# 6: Clean text

In [6]:
def clean_text(text):
    text = str(text)
    text = text.strip()
    text = re.sub(r"\s+", " ", text)
    text = text.replace("\u200c", "")
    text = text.replace("\u200d", "")
    return text

df["offensive_text"] = df["offensive_text"].apply(clean_text)
df["counterspeech_text"] = df["counterspeech_text"].apply(clean_text)

df = df.dropna()
df = df.drop_duplicates()
df = df[df["offensive_text"].str.len() > 2]
df = df[df["counterspeech_text"].str.len() > 2]

print("Cleaned dataset shape:", df.shape)
df.head()

Cleaned dataset shape: (2506, 2)


,offensive_text,counterspeech_text
0,"কি মনে করছ, মেয়েরা কিছু বোঝে ?",এই কথা ঠিক না। বাংলাদেশে অনেক মেয়ে দক্ষতার সাথ...
1,"তুই তো বেশ্যা, রাস্তায় খেপ মাইরা বেরাছ",মানুষ কে সম্মান করতে শিখুন। তবেই নিজে সম্মানিত...
2,হিন্দুরাই দেশের সম্পদ পাচার করতেছে,পুরো একটা গোষ্ঠীকে দোষ দেওয়া ঠিক না।
3,মাইয়া মানুষ আবার গাড়ি চালাইতে পারে নাকি!,এই ধারণা ভুল। বর্তমানে অনেক মেয়েরা সফলভাবে গাড়...
4,এই ছাগলের কথা শুনলে মাথাই গরম হয়ে যায়,ভদ্রভাবে কথা বললে আলোচনা আরও সুন্দর হয়।


# 7: Create input and target text

In [7]:
TASK_PREFIX = "আপত্তিকর বাংলা বক্তব্যের জন্য একটি ভদ্র, নিরাপদ, অ-আক্রমণাত্মক এবং গালিবিহীন বাংলা প্রতিউত্তর লিখুন: "

df["input_text"] = TASK_PREFIX + df["offensive_text"]
df["target_text"] = df["counterspeech_text"]

df = df[["input_text", "target_text"]]

df.head()

,input_text,target_text
0,"আপত্তিকর বাংলা বক্তব্যের জন্য একটি ভদ্র, নিরাপ...",এই কথা ঠিক না। বাংলাদেশে অনেক মেয়ে দক্ষতার সাথ...
1,"আপত্তিকর বাংলা বক্তব্যের জন্য একটি ভদ্র, নিরাপ...",মানুষ কে সম্মান করতে শিখুন। তবেই নিজে সম্মানিত...
2,"আপত্তিকর বাংলা বক্তব্যের জন্য একটি ভদ্র, নিরাপ...",পুরো একটা গোষ্ঠীকে দোষ দেওয়া ঠিক না।
3,"আপত্তিকর বাংলা বক্তব্যের জন্য একটি ভদ্র, নিরাপ...",এই ধারণা ভুল। বর্তমানে অনেক মেয়েরা সফলভাবে গাড়...
4,"আপত্তিকর বাংলা বক্তব্যের জন্য একটি ভদ্র, নিরাপ...",ভদ্রভাবে কথা বললে আলোচনা আরও সুন্দর হয়।


# 8: Train-validation-test split

In [8]:
train_df, temp_df = train_test_split(
    df,
    test_size=0.20,
    random_state=SEED,
    shuffle=True
)

valid_df, test_df = train_test_split(
    temp_df,
    test_size=0.50,
    random_state=SEED,
    shuffle=True
)

print("Train size:", len(train_df))
print("Validation size:", len(valid_df))
print("Test size:", len(test_df))

Train size: 2004
Validation size: 251
Test size: 251


# 9: Convert to Hugging Face Dataset

In [9]:
dataset = DatasetDict({
    "train": Dataset.from_pandas(train_df.reset_index(drop=True)),
    "validation": Dataset.from_pandas(valid_df.reset_index(drop=True)),
    "test": Dataset.from_pandas(test_df.reset_index(drop=True))
})

dataset

DatasetDict({
    train: Dataset({
        features: ['input_text', 'target_text'],
        num_rows: 2004
    })
    validation: Dataset({
        features: ['input_text', 'target_text'],
        num_rows: 251
    })
    test: Dataset({
        features: ['input_text', 'target_text'],
        num_rows: 251
    })
})

# 10: Load mT5-small

In [10]:
MODEL_NAME = "google/mt5-small"

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, use_fast=False)
model = AutoModelForSeq2SeqLM.from_pretrained(MODEL_NAME)

model.config.use_cache = False

if model.config.decoder_start_token_id is None:
    model.config.decoder_start_token_id = tokenizer.pad_token_id

model.to(device)

print("Model:", MODEL_NAME)
print("Tokenizer vocab size:", len(tokenizer))
print("Model vocab size:", model.config.vocab_size)
print("Pad token:", tokenizer.pad_token, tokenizer.pad_token_id)
print("EOS token:", tokenizer.eos_token, tokenizer.eos_token_id)
print("Decoder start token id:", model.config.decoder_start_token_id)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:112: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/553 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/82.0 [00:00<?, ?B/s]

spiece.model:   0%|          | 0.00/4.31M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/99.0 [00:00<?, ?B/s]

...

Loading weights:   0%|          | 0/192 [00:00<?, ?it/s]

[transformers] The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints with different values, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning.


generation_config.json:   0%|          | 0.00/147 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/1.20G [00:00<?, ?B/s]

Model: google/mt5-small
Tokenizer vocab size: 250100
Model vocab size: 250112
Pad token: <pad> 0
EOS token: </s> 1
Decoder start token id: 0


# 11: Tokenize dataset

In [11]:
MAX_INPUT_LENGTH = 128
MAX_TARGET_LENGTH = 96

def preprocess_function(examples):
    inputs = examples["input_text"]
    targets = examples["target_text"]

    model_inputs = tokenizer(
        inputs,
        max_length=MAX_INPUT_LENGTH,
        padding="max_length",
        truncation=True
    )

    target_tokens = tokenizer(
        targets,
        max_length=MAX_TARGET_LENGTH,
        padding="max_length",
        truncation=True
    )

    labels = target_tokens["input_ids"]

    labels = [
        [(token if token != tokenizer.pad_token_id else -100) for token in label]
        for label in labels
    ]

    model_inputs["labels"] = labels

    return model_inputs


tokenized_dataset = dataset.map(
    preprocess_function,
    batched=True,
    remove_columns=["input_text", "target_text"]
)

tokenized_dataset

Map:   0%|          | 0/2004 [00:00<?, ? examples/s]

Map:   0%|          | 0/251 [00:00<?, ? examples/s]

Map:   0%|          | 0/251 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['input_ids', 'attention_mask', 'labels'],
        num_rows: 2004
    })
    validation: Dataset({
        features: ['input_ids', 'attention_mask', 'labels'],
        num_rows: 251
    })
    test: Dataset({
        features: ['input_ids', 'attention_mask', 'labels'],
        num_rows: 251
    })
})

# 12: Debug tokenization

In [12]:
sample = tokenized_dataset["train"][0]

decoded_input = tokenizer.decode(
    [x for x in sample["input_ids"] if x != tokenizer.pad_token_id],
    skip_special_tokens=True
)

decoded_label = tokenizer.decode(
    [x for x in sample["labels"] if x != -100],
    skip_special_tokens=True
)

print("Decoded input:")
print(decoded_input)

print("\nDecoded label:")
print(decoded_label)

valid_label_tokens = [x for x in sample["labels"] if x != -100]
print("\nValid label token count:", len(valid_label_tokens))

Decoded input:
আপত্তিকর বাংলা বক্তব্যের জন্য একটি ভদ্র, নিরাপদ, অ-আক্রমণাত্মক এবং গালিবিহীন বাংলা প্রতিউত্তর লিখুন: আমাদের দেশের হিজরাই তোর মত মাগির জন্য যথেষ্ট

Decoded label:
হিজড়া পরিচয় নিয়ে বিদ্রূপ করা ঠিক নয়, সবার প্রতি সম্মান থাকা দরকার।

Valid label token count: 28


# 13: Data collator

In [13]:
data_collator = DataCollatorForSeq2Seq(
    tokenizer=tokenizer,
    model=model,
    padding=True,
    label_pad_token_id=-100
)

# 14: Batch loss test

In [14]:
from torch.utils.data import DataLoader

small_train_dataset = tokenized_dataset["train"].select(range(8))

test_loader = DataLoader(
    small_train_dataset,
    batch_size=2,
    collate_fn=data_collator
)

batch = next(iter(test_loader))
batch = {k: v.to(device) for k, v in batch.items()}

print("Batch input_ids shape:", batch["input_ids"].shape)
print("Batch labels shape:", batch["labels"].shape)

non_ignored_labels = (batch["labels"] != -100).sum().item()
print("Non-ignored label tokens:", non_ignored_labels)

with torch.no_grad():
    outputs = model(**batch)

print("Test batch loss:", outputs.loss.item())

Batch input_ids shape: torch.Size([2, 128])
Batch labels shape: torch.Size([2, 96])
Non-ignored label tokens: 51
Test batch loss: 20.70123863220215


# 15: Training arguments

In [15]:
OUTPUT_DIR = "/content/mt5_small_counterspeech_model"

training_args = Seq2SeqTrainingArguments(
    output_dir=OUTPUT_DIR,

    eval_strategy="epoch",
    save_strategy="epoch",

    learning_rate=3e-5,

    per_device_train_batch_size=4,
    per_device_eval_batch_size=4,

    gradient_accumulation_steps=4,

    num_train_epochs=12,

    weight_decay=0.01,

    predict_with_generate=False,

    fp16=False,

    logging_steps=25,

    save_total_limit=2,

    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    greater_is_better=False,

    report_to="none",

    seed=SEED
)

# 16: Create trainer

In [16]:
trainer = Seq2SeqTrainer(
    model=model,
    args=training_args,

    train_dataset=tokenized_dataset["train"],
    eval_dataset=tokenized_dataset["validation"],

    processing_class=tokenizer,
    data_collator=data_collator,

    callbacks=[
        EarlyStoppingCallback(early_stopping_patience=3)
    ]
)

# 17: Train mT5-small

In [17]:
trainer.train()

Epoch,Training Loss,Validation Loss
1,38.366001,4.860095
2,21.544302,3.486776
3,16.606077,2.869439
4,14.544916,2.602276
5,13.303307,2.451223
6,12.688191,2.355428
7,12.411858,2.298818
8,12.082504,2.266589
9,11.727283,2.238718
10,11.579100,2.220487


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

[transformers] There were missing keys in the checkpoint model loaded: ['encoder.embed_tokens.weight', 'decoder.embed_tokens.weight'].


TrainOutput(global_step=1512, training_loss=18.007026652179697, metrics={'train_runtime': 2377.0402, 'train_samples_per_second': 10.117, 'train_steps_per_second': 0.636, 'total_flos': 3178843141570560.0, 'train_loss': 18.007026652179697, 'epoch': 12.0})

# 18: Evaluate loss

In [18]:
trainer.evaluate()

Training Loss,Validation Loss,Epoch
11.432858,2.208716,12


{'eval_loss': 2.20871639251709}

# 19: Save model

In [19]:
FINAL_MODEL_DIR = "/content/final_mt5_small_counterspeech"

trainer.save_model(FINAL_MODEL_DIR)
tokenizer.save_pretrained(FINAL_MODEL_DIR)

print("Model saved to:", FINAL_MODEL_DIR)

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Model saved to: /content/final_mt5_small_counterspeech


# 20: Manual generation test

In [20]:
model.eval()

def generate_counterspeech(text):
    inputs = tokenizer(
        text,
        return_tensors="pt",
        max_length=128,
        truncation=True
    ).to(device)

    with torch.no_grad():
        output_ids = model.generate(
            **inputs,
            max_new_tokens=70,
            min_new_tokens=6,

            num_beams=5,
            no_repeat_ngram_size=2,
            repetition_penalty=1.15,
            length_penalty=0.9,

            early_stopping=True
        )

    return tokenizer.decode(output_ids[0], skip_special_tokens=True).strip()


for i in range(5):
    text = test_df["input_text"].iloc[i]
    ref = test_df["target_text"].iloc[i]
    pred = generate_counterspeech(text)

    print("Input:", text)
    print("Reference:", ref)
    print("Generated:", pred)
    print("Generated length:", len(pred.split()))
    print("-" * 80)

Input: আপত্তিকর বাংলা বক্তব্যের জন্য একটি ভদ্র, নিরাপদ, অ-আক্রমণাত্মক এবং গালিবিহীন বাংলা প্রতিউত্তর লিখুন: ঠিক বলছেন ভাই ওই শালা একটা পাগল চোদা
Reference: কাউকে পাগল বলে উড়িয়ে না দিয়ে কথা দিয়ে সমালোচনা করা উচিত।
Generated: কারও কথা বলা খুবই অপমানজনক।
Generated length: 5
--------------------------------------------------------------------------------
Input: আপত্তিকর বাংলা বক্তব্যের জন্য একটি ভদ্র, নিরাপদ, অ-আক্রমণাত্মক এবং গালিবিহীন বাংলা প্রতিউত্তর লিখুন: বাংলাদেশ বলে কথা এবং কিছু মুর্খ বলে এই চেনেলটি কোন দেশের মুর্খ মাদ্রাসা ছাপ ভালো করে দেখ
Reference: চ্যানেল বা উৎস নিয়ে সন্দেহ থাকলে অপমান না করে যাচাই করা তথ্য দিন।
Generated: কারও সমালোচনা করা উচিত।
Generated length: 4
--------------------------------------------------------------------------------
Input: আপত্তিকর বাংলা বক্তব্যের জন্য একটি ভদ্র, নিরাপদ, অ-আক্রমণাত্মক এবং গালিবিহীন বাংলা প্রতিউত্তর লিখুন: এই মাদারচোদ বাংলাদেশ ইন্ডিয়া এদের নিয়া এই সব ভিডিও করে খানকির ছাওয়াল এ
Reference: কাউকে পরিবার টেনে গালি না দিয়ে ভিডিওর বিষয়ট

# 21: Generate full test predictions

In [21]:
test_inputs = test_df["input_text"].tolist()
references = test_df["target_text"].tolist()

predictions = []

for i, text in enumerate(test_inputs):
    pred = generate_counterspeech(text)
    predictions.append(pred)

    if i % 50 == 0:
        print(f"Generated {i}/{len(test_inputs)}")

results_df = pd.DataFrame({
    "input_text": test_inputs,
    "reference_counterspeech": references,
    "generated_counterspeech": predictions
})

results_df["generated_length"] = results_df["generated_counterspeech"].apply(lambda x: len(str(x).split()))

results_df.head(10)

Generated 0/251
Generated 50/251
Generated 100/251
Generated 150/251
Generated 200/251
Generated 250/251


,input_text,reference_counterspeech,generated_counterspeech,generated_length
0,"আপত্তিকর বাংলা বক্তব্যের জন্য একটি ভদ্র, নিরাপ...",কাউকে পাগল বলে উড়িয়ে না দিয়ে কথা দিয়ে সমালোচনা...,কারও কথা বলা খুবই অপমানজনক।,5
1,"আপত্তিকর বাংলা বক্তব্যের জন্য একটি ভদ্র, নিরাপ...",চ্যানেল বা উৎস নিয়ে সন্দেহ থাকলে অপমান না করে...,কারও সমালোচনা করা উচিত।,4
2,"আপত্তিকর বাংলা বক্তব্যের জন্য একটি ভদ্র, নিরাপ...",কাউকে পরিবার টেনে গালি না দিয়ে ভিডিওর বিষয়টি ভ...,কারও কথা বলা খুবই অপমানজনক।,5
3,"আপত্তিকর বাংলা বক্তব্যের জন্য একটি ভদ্র, নিরাপ...","কাউকে ধরতে বা থ্যাড়াপি দিতে বলা হুমকি, যা পরিস...",কারও ছেলেকে ধর্ষণের কথা বলা উচিত।,6
4,"আপত্তিকর বাংলা বক্তব্যের জন্য একটি ভদ্র, নিরাপ...",রাজনৈতিক ব্যঙ্গের সাথে পরিবার ও নারীদের টেনে অ...,কারও কথা বলা উচিত।,4
5,"আপত্তিকর বাংলা বক্তব্যের জন্য একটি ভদ্র, নিরাপ...",বিরোধ থাকলেও কাউকে তুচ্ছ করে কথা না বলাই ভালো।...,কারও সমালোচনা করা উচিত।,4
6,"আপত্তিকর বাংলা বক্তব্যের জন্য একটি ভদ্র, নিরাপ...","পোস্টটি কী বোঝাচ্ছে আগে পরিষ্কার করা দরকার, তা...",কারও সমালোচনা করা উচিত।,4
7,"আপত্তিকর বাংলা বক্তব্যের জন্য একটি ভদ্র, নিরাপ...",কোনো গোষ্ঠীর বিরুদ্ধে হিংসার কথা না বলে শান্তভ...,কারও সমালোচনা করা উচিত।,4
8,"আপত্তিকর বাংলা বক্তব্যের জন্য একটি ভদ্র, নিরাপ...",কাউকে লাথি মেরে বের করার কথা বলা সহিংসতা বাড়ায়...,কারও পরিবারকে অপমান করা খুবই কষ্টদায়ক।,6
9,"আপত্তিকর বাংলা বক্তব্যের জন্য একটি ভদ্র, নিরাপ...",সাধারণীকরণ করলে আলোচনা দুর্বল হয় তাই যুক্তি দি...,কারও সমালোচনা করা উচিত।,4


# 22: Check generated length

In [22]:
print(results_df["generated_length"].describe())

results_df[["reference_counterspeech", "generated_counterspeech", "generated_length"]].head(20)

count    251.000000
mean       5.000000
std        0.971597
min        4.000000
25%        4.000000
50%        5.000000
75%        5.000000
max        8.000000
Name: generated_length, dtype: float64


,reference_counterspeech,generated_counterspeech,generated_length
0,কাউকে পাগল বলে উড়িয়ে না দিয়ে কথা দিয়ে সমালোচনা...,কারও কথা বলা খুবই অপমানজনক।,5
1,চ্যানেল বা উৎস নিয়ে সন্দেহ থাকলে অপমান না করে...,কারও সমালোচনা করা উচিত।,4
2,কাউকে পরিবার টেনে গালি না দিয়ে ভিডিওর বিষয়টি ভ...,কারও কথা বলা খুবই অপমানজনক।,5
3,"কাউকে ধরতে বা থ্যাড়াপি দিতে বলা হুমকি, যা পরিস...",কারও ছেলেকে ধর্ষণের কথা বলা উচিত।,6
4,রাজনৈতিক ব্যঙ্গের সাথে পরিবার ও নারীদের টেনে অ...,কারও কথা বলা উচিত।,4
5,বিরোধ থাকলেও কাউকে তুচ্ছ করে কথা না বলাই ভালো।...,কারও সমালোচনা করা উচিত।,4
6,"পোস্টটি কী বোঝাচ্ছে আগে পরিষ্কার করা দরকার, তা...",কারও সমালোচনা করা উচিত।,4
7,কোনো গোষ্ঠীর বিরুদ্ধে হিংসার কথা না বলে শান্তভ...,কারও সমালোচনা করা উচিত।,4
8,কাউকে লাথি মেরে বের করার কথা বলা সহিংসতা বাড়ায়...,কারও পরিবারকে অপমান করা খুবই কষ্টদায়ক।,6
9,সাধারণীকরণ করলে আলোচনা দুর্বল হয় তাই যুক্তি দি...,কারও সমালোচনা করা উচিত।,4


# 23: Save predictions

In [23]:
prediction_csv = "/content/mt5_small_test_predictions.csv"

results_df.to_csv(prediction_csv, index=False, encoding="utf-8-sig")

print("Saved predictions:", prediction_csv)

files.download(prediction_csv)

Saved predictions: /content/mt5_small_test_predictions.csv


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

# 24: Metric helper functions

In [24]:
smooth = SmoothingFunction().method1

def tokenize_bn(text):
    text = str(text).strip()
    text = re.sub(r"\s+", " ", text)
    return text.split()


def calculate_bleu_scores(predictions, references):
    bleu_1_scores = []
    bleu_2_scores = []
    bleu_3_scores = []
    bleu_4_scores = []

    for pred, ref in zip(predictions, references):
        pred_tokens = tokenize_bn(pred)
        ref_tokens = tokenize_bn(ref)

        if len(pred_tokens) == 0 or len(ref_tokens) == 0:
            bleu_1_scores.append(0)
            bleu_2_scores.append(0)
            bleu_3_scores.append(0)
            bleu_4_scores.append(0)
            continue

        bleu_1_scores.append(sentence_bleu(
            [ref_tokens],
            pred_tokens,
            weights=(1, 0, 0, 0),
            smoothing_function=smooth
        ))

        bleu_2_scores.append(sentence_bleu(
            [ref_tokens],
            pred_tokens,
            weights=(0.5, 0.5, 0, 0),
            smoothing_function=smooth
        ))

        bleu_3_scores.append(sentence_bleu(
            [ref_tokens],
            pred_tokens,
            weights=(1/3, 1/3, 1/3, 0),
            smoothing_function=smooth
        ))

        bleu_4_scores.append(sentence_bleu(
            [ref_tokens],
            pred_tokens,
            weights=(0.25, 0.25, 0.25, 0.25),
            smoothing_function=smooth
        ))

    return {
        "BLEU-1": float(np.mean(bleu_1_scores)),
        "BLEU-2": float(np.mean(bleu_2_scores)),
        "BLEU-3": float(np.mean(bleu_3_scores)),
        "BLEU-4": float(np.mean(bleu_4_scores))
    }


def calculate_meteor(predictions, references):
    scores = []

    for pred, ref in zip(predictions, references):
        pred_tokens = tokenize_bn(pred)
        ref_tokens = tokenize_bn(ref)

        if len(pred_tokens) == 0 or len(ref_tokens) == 0:
            scores.append(0)
            continue

        try:
            score = meteor_score([ref_tokens], pred_tokens)
        except:
            score = 0

        scores.append(score)

    return float(np.mean(scores))


def get_ngrams(tokens, n):
    return list(zip(*[tokens[i:] for i in range(n)]))


def calculate_diversity(predictions):
    all_unigrams = []
    all_bigrams = []

    for text in predictions:
        tokens = tokenize_bn(text)
        all_unigrams.extend(tokens)
        all_bigrams.extend(get_ngrams(tokens, 2))

    distinct_1 = len(set(all_unigrams)) / len(all_unigrams) if len(all_unigrams) > 0 else 0
    distinct_2 = len(set(all_bigrams)) / len(all_bigrams) if len(all_bigrams) > 0 else 0

    diversity = (distinct_1 + distinct_2) / 2

    return {
        "Distinct-1": distinct_1,
        "Distinct-2": distinct_2,
        "Diversity": diversity
    }


def normalize_bn_text(text):
    text = str(text).strip()
    text = re.sub(r"\s+", " ", text)
    return text


def token_jaccard_similarity(a, b):
    a_tokens = set(tokenize_bn(normalize_bn_text(a)))
    b_tokens = set(tokenize_bn(normalize_bn_text(b)))

    if len(a_tokens) == 0 or len(b_tokens) == 0:
        return 0.0

    intersection = len(a_tokens.intersection(b_tokens))
    union = len(a_tokens.union(b_tokens))

    return intersection / union if union > 0 else 0.0


def calculate_novelty(predictions, train_targets, similarity_threshold=0.70):
    train_targets = [normalize_bn_text(x) for x in train_targets]
    predictions = [normalize_bn_text(x) for x in predictions]

    max_similarities = []
    near_copy_count = 0

    for pred in predictions:
        if len(pred.strip()) == 0:
            max_similarities.append(0.0)
            continue

        similarities = [
            token_jaccard_similarity(pred, train_text)
            for train_text in train_targets
        ]

        max_similarity = max(similarities)
        max_similarities.append(max_similarity)

        if max_similarity >= similarity_threshold:
            near_copy_count += 1

    average_max_similarity = float(np.mean(max_similarities)) if len(max_similarities) > 0 else 0.0
    near_copy_rate = near_copy_count / len(predictions) if len(predictions) > 0 else 0.0

    novelty = 1 - average_max_similarity

    return {
        "Novelty": novelty,
        "Average Max Train Similarity": average_max_similarity,
        "Near Copy Rate": near_copy_rate
    }


bangla_abuse_words = [
    "চুদ",
    "মাগী",
    "হারামি",
    "শালা",
    "কুত্তা",
    "বাল",
    "খানকি",
    "বেশ্যা",
    "মাদারচোদ",
    "বোকাচোদা"
]


def calculate_abuse_word_rate(predictions, abuse_words):
    count = 0

    for text in predictions:
        text = str(text)
        found = any(word in text for word in abuse_words)

        if found:
            count += 1

    abuse_rate = count / len(predictions) if len(predictions) > 0 else 0

    return {
        "Abuse-word rate": abuse_rate,
        "Abusive outputs": count
    }


def calculate_length_stats(predictions):
    lengths = [len(tokenize_bn(text)) for text in predictions]

    empty_count = sum(1 for text in predictions if len(str(text).strip()) == 0)
    empty_rate = empty_count / len(predictions) if len(predictions) > 0 else 0

    return {
        "Average Generated Length": float(np.mean(lengths)),
        "Empty Output Rate": empty_rate
    }

# 25: Bengali ROUGE

In [25]:
def simple_tokenize_bn(text):
    text = str(text).strip()
    text = re.sub(r"\s+", " ", text)
    return text.split()


def get_ngrams_list(tokens, n):
    if len(tokens) < n:
        return []
    return [tuple(tokens[i:i+n]) for i in range(len(tokens)-n+1)]


def rouge_n_score(pred, ref, n=1):
    pred_tokens = simple_tokenize_bn(pred)
    ref_tokens = simple_tokenize_bn(ref)

    pred_ngrams = get_ngrams_list(pred_tokens, n)
    ref_ngrams = get_ngrams_list(ref_tokens, n)

    if len(pred_ngrams) == 0 or len(ref_ngrams) == 0:
        return 0.0

    pred_counts = {}
    ref_counts = {}

    for ng in pred_ngrams:
        pred_counts[ng] = pred_counts.get(ng, 0) + 1

    for ng in ref_ngrams:
        ref_counts[ng] = ref_counts.get(ng, 0) + 1

    overlap = 0

    for ng in ref_counts:
        overlap += min(ref_counts[ng], pred_counts.get(ng, 0))

    recall = overlap / len(ref_ngrams)
    precision = overlap / len(pred_ngrams)

    if precision + recall == 0:
        return 0.0

    return 2 * precision * recall / (precision + recall)


def lcs_length(x, y):
    m, n = len(x), len(y)
    dp = [[0] * (n + 1) for _ in range(m + 1)]

    for i in range(m):
        for j in range(n):
            if x[i] == y[j]:
                dp[i + 1][j + 1] = dp[i][j] + 1
            else:
                dp[i + 1][j + 1] = max(dp[i][j + 1], dp[i + 1][j])

    return dp[m][n]


def rouge_l_score(pred, ref):
    pred_tokens = simple_tokenize_bn(pred)
    ref_tokens = simple_tokenize_bn(ref)

    if len(pred_tokens) == 0 or len(ref_tokens) == 0:
        return 0.0

    lcs = lcs_length(pred_tokens, ref_tokens)

    recall = lcs / len(ref_tokens)
    precision = lcs / len(pred_tokens)

    if precision + recall == 0:
        return 0.0

    return 2 * precision * recall / (precision + recall)


def calculate_custom_rouge(predictions, references):
    rouge1_scores = []
    rouge2_scores = []
    rougeL_scores = []

    for pred, ref in zip(predictions, references):
        rouge1_scores.append(rouge_n_score(pred, ref, n=1))
        rouge2_scores.append(rouge_n_score(pred, ref, n=2))
        rougeL_scores.append(rouge_l_score(pred, ref))

    return {
        "rouge1": float(np.mean(rouge1_scores)),
        "rouge2": float(np.mean(rouge2_scores)),
        "rougeL": float(np.mean(rougeL_scores))
    }


rouge_scores = calculate_custom_rouge(predictions, references)

rouge_scores

{'rouge1': 0.13222654572374817,
 'rouge2': 0.03337322712755964,
 'rougeL': 0.1305832876565292}

# 26: BERTScore

In [26]:
bertscore = evaluate.load("bertscore")

bertscore_result = bertscore.compute(
    predictions=predictions,
    references=references,
    model_type="xlm-roberta-base",
    lang="bn",
    verbose=True
)

bertscore_precision = float(np.mean(bertscore_result["precision"]))
bertscore_recall = float(np.mean(bertscore_result["recall"]))
bertscore_f1 = float(np.mean(bertscore_result["f1"]))

print("BERTScore Precision:", bertscore_precision)
print("BERTScore Recall:", bertscore_recall)
print("BERTScore F1:", bertscore_f1)

config.json:   0%|          | 0.00/615 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

sentencepiece.bpe.model:   0%|          | 0.00/5.07M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/9.10M [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/1.12G [00:00<?, ?B/s]

...

calculating scores...
computing bert embedding.


  0%|          | 0/5 [00:00<?, ?it/s]

computing greedy matching.


  0%|          | 0/4 [00:00<?, ?it/s]

done in 4762.58 seconds, 0.05 sentences/sec
BERTScore Precision: 0.8866634053063107
BERTScore Recall: 0.848307438105701
BERTScore F1: 0.8670005748471415


# 27: Calculate final metrics

In [27]:
bleu_scores = calculate_bleu_scores(predictions, references)

meteor = calculate_meteor(predictions, references)

diversity_scores = calculate_diversity(predictions)

novelty_scores = calculate_novelty(
    predictions,
    train_df["target_text"].tolist(),
    similarity_threshold=0.70
)

abuse_scores = calculate_abuse_word_rate(
    predictions,
    bangla_abuse_words
)

length_scores = calculate_length_stats(predictions)

final_metrics = {
    "Model": "mT5-base",

    "BLEU-1": bleu_scores["BLEU-1"],
    "BLEU-2": bleu_scores["BLEU-2"],
    "BLEU-3": bleu_scores["BLEU-3"],
    "BLEU-4": bleu_scores["BLEU-4"],

    "ROUGE-1": rouge_scores["rouge1"],
    "ROUGE-2": rouge_scores["rouge2"],
    "ROUGE-L": rouge_scores["rougeL"],

    "BERTScore Precision": bertscore_precision,
    "BERTScore Recall": bertscore_recall,
    "BERTScore F1": bertscore_f1,

    "Diversity": diversity_scores["Diversity"],
    "Distinct-1": diversity_scores["Distinct-1"],
    "Distinct-2": diversity_scores["Distinct-2"],

    "Novelty": novelty_scores["Novelty"],
    "Average Max Train Similarity": novelty_scores["Average Max Train Similarity"],
    "Near Copy Rate": novelty_scores["Near Copy Rate"],

    "Abuse-word rate": abuse_scores["Abuse-word rate"],
    "Abusive outputs": abuse_scores["Abusive outputs"],

    "METEOR": meteor,

    "Average Generated Length": length_scores["Average Generated Length"],
    "Empty Output Rate": length_scores["Empty Output Rate"]
}

metrics_df = pd.DataFrame([final_metrics])

metrics_df

,Model,BLEU-1,BLEU-2,BLEU-3,BLEU-4,ROUGE-1,ROUGE-2,ROUGE-L,BERTScore Precision,BERTScore Recall,...,Distinct-1,Distinct-2,Novelty,Average Max Train Similarity,Near Copy Rate,Abuse-word rate,Abusive outputs,METEOR,Average Generated Length,Empty Output Rate
0,mT5-base,0.061865,0.03169,0.020151,0.015879,0.132227,0.033373,0.130583,0.886663,0.848307,...,0.047012,0.110558,0.546332,0.453668,0.0,0.003984,1,0.06965,5.0,0.0


# 28: Save metrics

In [28]:
metrics_csv = "/content/mt5_small_final_metrics.csv"

metrics_df.to_csv(metrics_csv, index=False, encoding="utf-8-sig")

print("Saved metrics:", metrics_csv)

files.download(metrics_csv)

Saved metrics: /content/mt5_small_final_metrics.csv


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

# 29: Zip and download model

In [29]:
!zip -r /content/final_mt5_small_counterspeech.zip /content/final_mt5_small_counterspeech

files.download("/content/final_mt5_small_counterspeech.zip")

  adding: content/final_mt5_small_counterspeech/ (stored 0%)
  adding: content/final_mt5_small_counterspeech/tokenizer.json (deflated 76%)
  adding: content/final_mt5_small_counterspeech/generation_config.json (deflated 27%)
  adding: content/final_mt5_small_counterspeech/tokenizer_config.json (deflated 84%)
  adding: content/final_mt5_small_counterspeech/model.safetensors (deflated 26%)
  adding: content/final_mt5_small_counterspeech/config.json (deflated 49%)
  adding: content/final_mt5_small_counterspeech/training_args.bin (deflated 53%)


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>